# C6 example 4/4: `RestrictedWETensorProduct` with internal spherical harmonics

This notebook keeps the same physical features and hidden architecture, but uses e3nn's 3D spherical harmonics restricted to the planar C6 subgroup of O(3). C6 acts by rotations around z. Thus each input/output 3D vector is still explicitly represented as $(x,y)\in E_1$ plus $z\in A$.

The important API difference is that `RestrictedWETensorProduct.forward_from_points` accepts the geometric 3D edge displacement directly. This is $r_{ij}=p_j-p_i$ in message passing, not a fourth node feature. The layer owns a `RestrictedSphericalHarmonics` evaluator and computes the filter internally. Without edge/filter geometry, this layer is unnecessary; use `WELinear` or an ordinary `TensorProduct` instead:

$$r\xrightarrow{Y_0\oplus\cdots\oplus Y_3}Y(r)\xrightarrow{\mathrm{RestrictedWETP}}\text{features}.$$

The default C6 full bandlimit is $L_{full}=3$, so degrees $l=0,1,2,3$ are included.

In [ ]:
import torch
from we3nn import CyclicGroup, RestrictedSphericalHarmonics, nn

torch.manual_seed(7)
torch.set_printoptions(precision=5, sci_mode=False)
G = CyclicGroup(6)
A = G.trivial_representation
E1 = G.standard_representation
regular = G.regular_representation()
input_rep = 3 * E1 + 5 * A
hidden_rep = 2 * regular
output_rep = E1 + 4 * A
spherical = RestrictedSphericalHarmonics(
    G, degrees=None, normalization='component', basis='o3'
)
print('spherical degrees:', spherical.degrees)
print('restricted filter representation:', spherical.rep_out.name)
print('dimensions:', input_rep.size, 'x', spherical.rep_out.size, '->', hidden_rep.size, '->', output_rep.size)

In [ ]:
def pack_input(vectors, scalars):
    xy = vectors[..., :, :2].reshape(*vectors.shape[:-2], 6)
    return torch.cat((xy, vectors[..., :, 2], scalars), dim=-1)

def unpack_input(x):
    xy = x[..., :6].reshape(*x.shape[:-1], 3, 2)
    return torch.cat((xy, x[..., 6:9].unsqueeze(-1)), dim=-1), x[..., 9:11]

def unpack_output(y):
    return torch.cat((y[..., :2], y[..., 2:3]), dim=-1), y[..., 3:6]

def rotate_points(points, element):
    Rxy = E1(element).to(device=points.device, dtype=points.dtype)
    return torch.cat((points[..., :2] @ Rxy.T, points[..., 2:3]), dim=-1)

vectors = torch.tensor([[[1.0, 0.2, -0.4], [-0.3, 0.8, 1.2], [0.5, -0.7, 0.1]]])
scalars = torch.tensor([[0.6, -1.1]])
# This is edge geometry r_ij, not a fourth node-feature vector.
edge_displacement = torch.tensor([[0.8, 0.35, -0.2]])
x = nn.RepresentationTensor(pack_input(vectors, scalars), input_rep)
Y = spherical(edge_displacement)
print('edge displacement r_ij:', edge_displacement)
print('internally used restricted spherical harmonics Y_l(r):', Y)
print('harmonic block sizes:', [2*l + 1 for l in spherical.degrees])

## Step 1: distinguish node features from edge geometry

The three colored arrows are node features. The dashed black arrow is the message-passing displacement $r_{ij}=p_j-p_i$. `RestrictedWETensorProduct` uses it to sample its filter, but it is not packed into `x` and is not a fourth node vector.

In [ ]:
import matplotlib.pyplot as plt

fig_input = plt.figure(figsize=(6, 5), constrained_layout=True)
ax = fig_input.add_subplot(111, projection='3d')
for index, vector in enumerate(vectors[0]):
    ax.quiver(0, 0, 0, *vector.tolist(), color=f'C{index}', linewidth=2, label=f'node feature v{index+1}')
ax.quiver(0, 0, 0, *edge_displacement[0].tolist(), color='black', linestyle='--', linewidth=2, label='edge displacement r_ij')
limit = 1.15 * torch.cat((vectors[0], edge_displacement)).abs().max().item()
ax.set(xlim=(-limit, limit), ylim=(-limit, limit), zlim=(-limit, limit), xlabel='x', ylabel='y', zlabel='z', title='Node features versus filter geometry')
ax.set_box_aspect((1, 1, 1)); ax.legend(fontsize=8)
print('scalar node features:', scalars[0].tolist())
plt.show()

## Step 2: see the spherical harmonics on their domain

Spherical harmonics are functions on $S^2$. The layer uses every real component in degrees $l=0,1,2,3$, for a total of $1+3+5+7=16$ filters. Each panel below is a standard signed-lobe view: radius is proportional to $|Y_{l,c}|$, red is positive, and blue is negative. The component index $c$ follows e3nn's real O(3) basis ordering.

In [ ]:
import math
import matplotlib.pyplot as plt
import matplotlib.colors as colors

polar = torch.linspace(0.0, math.pi, 25)
azimuth = torch.linspace(0.0, 2.0 * math.pi, 49)
polar_grid, azimuth_grid = torch.meshgrid(polar, azimuth, indexing='ij')
unit_sphere = torch.stack((
    torch.sin(polar_grid) * torch.cos(azimuth_grid),
    torch.sin(polar_grid) * torch.sin(azimuth_grid),
    torch.cos(polar_grid),
), dim=-1)
Y_sphere = spherical(unit_sphere).detach()

fig_sphere = plt.figure(figsize=(15, 15), constrained_layout=True)
sphere_axes = []
offset = 0
panel = 1
for degree in spherical.degrees:
    for component in range(2 * degree + 1):
        ax = fig_sphere.add_subplot(4, 4, panel, projection='3d')
        values = Y_sphere[..., offset + component]
        magnitude = values.abs() / values.abs().max().clamp_min(1e-8)
        surface = unit_sphere * magnitude.unsqueeze(-1)
        facecolors = plt.cm.coolwarm((values / values.abs().max().clamp_min(1e-8) + 1.0) / 2.0)
        ax.plot_surface(surface[..., 0].numpy(), surface[..., 1].numpy(), surface[..., 2].numpy(), facecolors=facecolors, linewidth=0, antialiased=False, shade=False)
        ax.set(xlim=(-1, 1), ylim=(-1, 1), zlim=(-1, 1), title=f'l={degree}, component={component}')
        ax.set_box_aspect((1, 1, 1)); ax.set_axis_off()
        sphere_axes.append(ax); panel += 1
    offset += 2 * degree + 1
colorbar = fig_sphere.colorbar(plt.cm.ScalarMappable(norm=colors.Normalize(-1, 1), cmap='coolwarm'), ax=sphere_axes, shrink=0.45, pad=0.02)
colorbar.set_label('sign of normalized harmonic value')
fig_sphere.suptitle('All 16 restricted spherical-harmonic components used by the layer', fontsize=16)
plt.show()

## Step 3: build the internally filtered architecture

The contraction has the same Wigner--Eckart form as notebook 3,

$$z_o=\sum_p w_p(\lVert r\rVert,r_z)(C_p)_{oij}x_iY_j(r),$$

but `forward_from_points` evaluates e3nn spherical harmonics inside the layer. The $C_p$ basis spans the complete finite-C6 Hom space after restriction; it is not limited to parent-O(3) coupling paths. The radial networks use only C6-invariant quantities.

In [ ]:
class RestrictedHarmonicNetwork(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.harmonics = spherical
        self.input_layer = nn.RestrictedWETensorProduct(
            input_rep, self.harmonics, hidden_rep, shared_weights=False
        )
        self.activation = nn.PointActiv(hidden_rep, torch.relu)
        self.output_layer = nn.RestrictedWETensorProduct(
            hidden_rep, self.harmonics, output_rep, shared_weights=False
        )
        self.radial_in = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.input_layer.weight_numel),
        )
        self.radial_out = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.SiLU(),
            torch.nn.Linear(16, self.output_layer.weight_numel),
        )

    def forward(self, features, edge_displacements):
        invariant_geometry = torch.stack(
            (torch.linalg.vector_norm(edge_displacements, dim=-1), edge_displacements[..., 2]), dim=-1
        )
        w_in = self.radial_in(invariant_geometry)
        w_out = self.radial_out(invariant_geometry)
        # Spherical filter evaluation happens inside both calls.
        h_pre = self.input_layer.forward_from_points(features, edge_displacements, w_in)
        h = self.activation(h_pre)
        y = self.output_layer.forward_from_points(h, edge_displacements, w_out)
        return y, w_in, w_out, h_pre, h

model = RestrictedHarmonicNetwork().eval()
y, w_in, w_out, h_pre, h = model(x, edge_displacement)
print('input/output reduced-weight counts:', model.input_layer.weight_numel, model.output_layer.weight_numel)
print('hidden before PointActiv:', h_pre.tensor)
print('hidden after  PointActiv:', h.tensor)
print('physical output (vector, scalars):', unpack_output(y.tensor))
kernel_basis = model.input_layer.sample_kernel_basis(edge_displacement)
print('sampled input-kernel basis shape [batch, paths, out, in]:', tuple(kernel_basis.shape))

## Step 4: rotate node features and edge geometry together

The input feature fiber and the 3D edge displacement are rotated together. We print the spherical harmonics at every rotated displacement, all physical inputs and outputs, and the direct equivariance residual.

In [ ]:
errors = []
for k, element in enumerate(G.elements):
    x_k = x.transform_fibers(element)
    edge_k = rotate_points(edge_displacement, element)
    y_k, w_in_k, w_out_k, _, _ = model(x_k, edge_k)
    expected_k = y.transform_fibers(element)
    Y_k = spherical(edge_k)
    error = (y_k.tensor - expected_k.tensor).abs().max().item()
    errors.append(error)
    torch.testing.assert_close(y_k.tensor, expected_k.tensor, atol=8e-5, rtol=8e-5)
    torch.testing.assert_close(w_in_k, w_in, atol=1e-6, rtol=1e-6)
    torch.testing.assert_close(w_out_k, w_out, atol=1e-6, rtol=1e-6)
    in_vectors_k, in_scalars_k = unpack_input(x_k.tensor)
    out_vector_k, out_scalars_k = unpack_output(y_k.tensor)
    print(f'rotation {k}: angle={60*k:3d} degrees')
    print('  rotated edge r_ij:', edge_k[0].tolist())
    print('  spherical Y     :', Y_k[0].tolist())
    print('  input vectors   :', in_vectors_k[0].tolist())
    print('  input scalars   :', in_scalars_k[0].tolist())
    print('  output vector   :', out_vector_k[0].tolist())
    print('  output scalars  :', out_scalars_k[0].tolist())
    print(f'  max equivariance error: {error:.3e}')

print('maximum over all rotations:', max(errors))

## Step 5: follow the spherical filter into the regular hidden state

The left heatmap samples all 16 spherical components at the six C6-related edge displacements and groups them by degree. The right heatmap shows how these filters and the rotated node features produce two cyclically permuted regular fields.

In [ ]:
import matplotlib.pyplot as plt

hidden_by_rotation, harmonic_by_rotation = [], []
output_vectors, output_scalars = [], []
for element in G.elements:
    x_k = x.transform_fibers(element)
    edge_k = rotate_points(edge_displacement, element)
    y_k, _, _, _, h_k = model(x_k, edge_k)
    vector_k, scalars_k = unpack_output(y_k.tensor)
    hidden_by_rotation.append(h_k.tensor[0].detach())
    harmonic_by_rotation.append(spherical(edge_k)[0].detach())
    output_vectors.append(vector_k[0].detach())
    output_scalars.append(scalars_k[0].detach())
hidden_by_rotation = torch.stack(hidden_by_rotation).cpu()
harmonic_by_rotation = torch.stack(harmonic_by_rotation).cpu()
output_vectors = torch.stack(output_vectors).cpu()
output_scalars = torch.stack(output_scalars).cpu()
angles_deg = torch.arange(6) * 60
colors = plt.cm.hsv(torch.linspace(0, 5/6, 6).numpy())
component_labels = [f'l={degree}[{component}]' for degree in spherical.degrees for component in range(2 * degree + 1)]
degree_boundaries = torch.tensor([sum(2 * degree + 1 for degree in spherical.degrees[:end]) for end in range(1, len(spherical.degrees))])

fig_state, (ax_harm, ax_hidden) = plt.subplots(1, 2, figsize=(16, 5), constrained_layout=True)
harmonic_image = ax_harm.imshow(harmonic_by_rotation, aspect='auto', cmap='coolwarm')
for boundary in degree_boundaries.tolist():
    ax_harm.axvline(boundary - 0.5, color='white', linewidth=2)
ax_harm.set(xticks=range(len(component_labels)), xticklabels=component_labels, yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='real spherical-harmonic component', ylabel='C6 rotation', title='Restricted spherical harmonics l=0,1,2,3')
ax_harm.tick_params(axis='x', labelrotation=90, labelsize=7)
fig_state.colorbar(harmonic_image, ax=ax_harm, shrink=0.75)

hidden_image = ax_hidden.imshow(hidden_by_rotation, aspect='auto', cmap='coolwarm')
ax_hidden.axvline(5.5, color='white', linewidth=2)
ax_hidden.set(xticks=range(12), yticks=range(6), yticklabels=[f'{a}°' for a in angles_deg.tolist()], xlabel='regular coordinate (copies 1 | 2)', ylabel='rotation', title='Hidden 2 Reg(C6) after PointActiv')
fig_state.colorbar(hidden_image, ax=ax_hidden, shrink=0.75)
plt.show()

## Step 6: inspect the restricted Wigner--Eckart kernel

The layer evaluates its owned spherical harmonics at $r_{ij}$, samples every finite-C6 kernel path $K_p(r)$, and contracts the path axis with invariant radial coefficients. The assertion verifies the explicit matrix operation $h_{pre}=K(r)x$.

In [ ]:
basis_at_edge = model.input_layer.sample_kernel_basis(edge_displacement)[0].detach()
effective_kernel = torch.einsum('p,poi->oi', w_in[0].detach(), basis_at_edge).cpu()
torch.testing.assert_close(h_pre.tensor[0], effective_kernel @ x.tensor[0], atol=3e-5, rtol=3e-5)

fig_kernel, ax_kernel = plt.subplots(figsize=(7, 5), constrained_layout=True)
kernel_image = ax_kernel.imshow(effective_kernel, aspect='auto', cmap='coolwarm')
ax_kernel.set(xlabel='input coordinate', ylabel='hidden coordinate', title='Restricted-WE effective kernel K(r)')
fig_kernel.colorbar(kernel_image, ax=ax_kernel, shrink=0.75)
plt.show()

## Step 7: unpack the final restricted-WE output

The final representation is one $xy$ vector plus four trivial coordinates. The first trivial coordinate is interpreted as vector $z$; the other three are scalar output features.

In [ ]:
fig_output = plt.figure(figsize=(12, 5), constrained_layout=True)
ax_vec = fig_output.add_subplot(1, 2, 1, projection='3d')
for k, (vector, color) in enumerate(zip(output_vectors, colors)):
    ax_vec.quiver(0, 0, 0, *vector.tolist(), color=color, linewidth=2, label=f'{60*k}°')
ax_vec.set(xlabel='x', ylabel='y', zlabel='z', title='RestrictedWETensorProduct output vector')
output_limit = max(1e-3, 1.15 * output_vectors.abs().max().item())
ax_vec.set_xlim(-output_limit, output_limit); ax_vec.set_ylim(-output_limit, output_limit); ax_vec.set_zlim(-output_limit, output_limit); ax_vec.set_box_aspect((1, 1, 1))
ax_vec.legend(ncols=2, fontsize=7)

ax_scalar = fig_output.add_subplot(1, 2, 2)
for channel in range(3):
    ax_scalar.plot(angles_deg, output_scalars[:, channel], marker='o', label=f'output scalar {channel+1}')
ax_scalar.set(xticks=angles_deg.tolist(), xlabel='C6 rotation', ylabel='value', title='Restricted-WE output scalars')
ax_scalar.grid(alpha=0.3); ax_scalar.legend()
plt.show()